# LTX-2.5 Distilled — Google Colab A100 Smoke Test

This notebook runs a minimal Text-to-Video + Synchronized Audio (T2V+A) smoke test using **Lightricks/LTX-2.5 Distilled** (8-step schedule).

**Before running:**
1. Select **Runtime → Change runtime type → A100 GPU** (High-RAM).
2. Accept the **LTX-2.x Community License Agreement** on [Lightricks/LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5).
3. Add your Hugging Face read token as `HF_TOKEN` in **Colab Secrets** (key icon on the left sidebar).

In [ ]:
#@title 1. Check GPU, Disk, and System RAM
!nvidia-smi
!df -h /content
!free -h
!python3 --version

In [ ]:
#@title 2. Mount Google Drive for Output Persistence
import os
from google.colab import drive

drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/ltx25_smoke_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Outputs will be saved to:', OUTPUT_DIR)

In [ ]:
#@title 3. Authenticate with Hugging Face (from Colab Secrets)
import os
from getpass import getpass
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = getpass('Enter your Hugging Face token (HF_TOKEN): ')

os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('Hugging Face login succeeded.')

In [ ]:
#@title 4. Clone Lightricks/LTX-2 & Install Dependencies via uv
%cd /content
!if [ ! -d LTX-2 ]; then git clone https://github.com/Lightricks/LTX-2.git; fi
%cd /content/LTX-2
!pip install -q uv
!uv python install 3.12
!uv sync --python 3.12
!uv sync --python 3.12 --inexact --package ltx-core --extra natten

In [ ]:
#@title 5. Download LTX-2.5 Distilled Weights Only (Selective Download)
import os
from huggingface_hub import hf_hub_download

REPO_ID = 'Lightricks/LTX-2.5'
MODEL_DIR = '/content/models/ltx-2.5'
os.makedirs(MODEL_DIR, exist_ok=True)

FILES = [
    'diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    'text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    'vae/ltx-2.5-video-vae-bf16.safetensors',
    'vae/ltx-2.5-audio-vae-bf16.safetensors',
    'model_patches/ltx-2.5-duration-head-bf16.safetensors',
    'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
]

for rel_path in FILES:
    print(f'Downloading {rel_path} ...')
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=rel_path,
        local_dir=MODEL_DIR,
    )
    print(f'  -> {local_path}')

!df -h /content

In [ ]:
#@title 6. Inspect Distilled Pipeline CLI Options
%cd /content/LTX-2
!MPLBACKEND=Agg uv run python -m ltx_pipelines.distilled --help

In [ ]:
#@title 7. Run Smoke Test (Short T2V + Synchronized Audio)
import os
import shutil
import subprocess
import time

PROMPT = 'A cinematic slow-motion shot of ocean waves crashing against dark rocky cliffs at sunset, with natural crashing wave audio.'
NUM_FRAMES = 49  # Must satisfy frames % 8 == 1 (~2s @ 24fps for fast smoke test)
LOCAL_OUT = '/content/smoke_test_ltx25.mp4'
DRIVE_OUT = '/content/drive/MyDrive/ltx25_smoke_test/smoke_test_ltx25.mp4'

cmd = [
    'uv', 'run', '--no-sync', 'python', '-m', 'ltx_pipelines.distilled',
    '--transformer-path', '/content/models/ltx-2.5/diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    '--text-encoder-path', '/content/models/ltx-2.5/text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    '--video-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-video-vae-bf16.safetensors',
    '--audio-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-audio-vae-bf16.safetensors',
    '--duration-head-path', '/content/models/ltx-2.5/model_patches/ltx-2.5-duration-head-bf16.safetensors',
    '--spatial-upsampler-path', '/content/models/ltx-2.5/latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
    '--num-frames', str(NUM_FRAMES),
    '--prompt', PROMPT,
    '--output-path', LOCAL_OUT,
]

env = {**os.environ, 'MPLBACKEND': 'Agg'}
t0 = time.time()
subprocess.run(cmd, cwd='/content/LTX-2', env=env, check=True)
elapsed = time.time() - t0
print(f'Smoke test completed in {elapsed:.1f}s')

shutil.copy2(LOCAL_OUT, DRIVE_OUT)
print('Saved output to Google Drive:', DRIVE_OUT)

In [ ]:
#@title 8. Check Peak GPU Memory & Preview Video
!nvidia-smi
from IPython.display import Video, display
display(Video('/content/smoke_test_ltx25.mp4', embed=True))

In [ ]:
#@title 9. Run ~1-Hour Talking-Head & Dialog Batch Test (Developer Explaining Diffusion Models)
import os
import shutil
import subprocess
import time

DRIVE_DIR = '/content/drive/MyDrive/ltx25_talking_head_1hr'
os.makedirs(DRIVE_DIR, exist_ok=True)

COMMON_ARGS = [
    'uv', 'run', '--no-sync', 'python', '-m', 'ltx_pipelines.distilled',
    '--transformer-path', '/content/models/ltx-2.5/diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    '--text-encoder-path', '/content/models/ltx-2.5/text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    '--video-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-video-vae-bf16.safetensors',
    '--audio-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-audio-vae-bf16.safetensors',
    '--duration-head-path', '/content/models/ltx-2.5/model_patches/ltx-2.5-duration-head-bf16.safetensors',
    '--spatial-upsampler-path', '/content/models/ltx-2.5/latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
]

JOBS = [
    {
        'name': 'dev_intro_5s',
        'num_frames': 121,
        'prompt': (
            'Medium close-up talking-head shot of a friendly software developer in a warmly lit home office '
            'with monitors and a whiteboard blurred in the background. Looking directly into the camera, '
            'speaking clearly with natural facial expressions and subtle hand gestures: '
            '"A diffusion model starts with pure random noise and learns to remove it step by step until a coherent image or video emerges." '
            'Clean studio microphone audio, no background music.'
        ),
    },
    {
        'name': 'dev_latents_10s',
        'num_frames': 241,
        'prompt': (
            'Medium shot of a female machine learning engineer sitting at a desk in a modern studio, '
            'looking at the camera and explaining a concept with natural lip-sync and expressive hand gestures: '
            '"Instead of predicting pixels directly, modern video models compress frames into a smaller latent space, '
            'denoise video and audio together, and then decode them back into high-resolution footage." '
            'Crisp podcast-quality voice audio, subtle room tone.'
        ),
    },
    {
        'name': 'dev_distilled_15s',
        'num_frames': 361,
        'prompt': (
            'Close-up talking-head video of a developer wearing glasses in a softly lit studio, '
            'speaking calmly and enthusiastically to the camera with natural eye contact and lip-sync: '
            '"The reason distilled models are so fast is that they compress fifty denoising steps down into just eight, '
            'so you can iterate on prompts and camera movement in minutes instead of waiting half an hour." '
            'Clear close-mic speech, natural breathing, no background noise.'
        ),
    },
]

MAX_RUNTIME_SECONDS = 3600
env = {**os.environ, 'MPLBACKEND': 'Agg'}
start_all = time.time()
run_idx = 0

while (time.time() - start_all) < MAX_RUNTIME_SECONDS:
    job = JOBS[run_idx % len(JOBS)]
    seed = 100 + run_idx
    out_name = f"{run_idx:03d}_{job['name']}_seed{seed}.mp4"
    local_out = f"/content/{out_name}"
    drive_out = f"{DRIVE_DIR}/{out_name}"

    cmd = COMMON_ARGS + [
        '--num-frames', str(job['num_frames']),
        '--seed', str(seed),
        '--prompt', job['prompt'],
        '--output-path', local_out,
    ]

    print(f"\n[{run_idx}] Starting {out_name} (elapsed: {(time.time() - start_all)/60:.1f} min)")
    t0 = time.time()
    subprocess.run(cmd, cwd='/content/LTX-2', env=env, check=True)
    dt = time.time() - t0
    shutil.copy2(local_out, drive_out)
    os.remove(local_out)
    print(f"[{run_idx}] Saved to Drive in {dt:.1f}s -> {drive_out}")
    run_idx += 1

print(f"\nDone. Generated {run_idx} clips in {(time.time() - start_all)/60:.1f} minutes.")